# Synthetic 3D Foot Scan Viewer (Colab)

This notebook generates synthetic left- and right-foot `.glb` illustrations, previews one in an interactive 3D viewer, and embeds the Firebase Hosting viewer page. It uses no real patient data. These models are not clinical scans, anatomical references, or diagnostic tools.

**Privacy:** Use synthetic examples only in hosted Colab. Do not upload identifiable or confidential patient scans. The generated examples are temporary Colab files unless you explicitly download them. The Firebase viewer page opens a file locally in your browser and does not upload it.

If Firebase Hosting has not been deployed yet, deploy from the repository root using the Firebase CLI: `firebase login`, then `firebase use coremedi-59bbc` and `firebase deploy --only hosting`.

In [ ]:
%pip -q install trimesh

from pathlib import Path
import json
import trimesh
from IPython.display import HTML, IFrame, display

OUT_DIR = Path('/content/foot_scan_demo')
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Demo output directory:', OUT_DIR)

## 1. Generate synthetic left and right foot models

In [ ]:
def ellipsoid(name, center, radii, color):
    mesh = trimesh.creation.icosphere(subdivisions=3, radius=1.0)
    mesh.apply_scale(radii)
    mesh.apply_translation(center)
    mesh.visual.face_colors = color
    mesh.metadata['name'] = name
    return mesh

def build_demo_foot(laterality):
    # Toes point toward +Y. Big toe is medial: +X for left and -X for right.
    skin, sole = (206, 154, 126, 255), (116, 77, 66, 255)
    sign = 1 if laterality == 'left' else -1
    toe_x = [sign * x for x in (0.049, 0.014, -0.018, -0.046, -0.068)]
    parts = [
        ellipsoid('sole-and-midfoot', (0.0, 0.015, 0.055), (0.060, 0.158, 0.043), sole),
        ellipsoid('heel', (0.0, -0.105, 0.074), (0.067, 0.070, 0.054), skin),
        ellipsoid('arch-and-instep', (0.0, -0.015, 0.100), (0.050, 0.112, 0.047), skin),
        ellipsoid('forefoot', (0.0, 0.105, 0.080), (0.086, 0.070, 0.045), skin),
        ellipsoid('toe-1-medial', (toe_x[0], 0.184, 0.071), (0.026, 0.055, 0.031), skin),
        ellipsoid('toe-2', (toe_x[1], 0.193, 0.069), (0.021, 0.050, 0.027), skin),
        ellipsoid('toe-3', (toe_x[2], 0.188, 0.067), (0.019, 0.044, 0.024), skin),
        ellipsoid('toe-4', (toe_x[3], 0.175, 0.065), (0.017, 0.037, 0.022), skin),
        ellipsoid('toe-5-lateral', (toe_x[4], 0.157, 0.063), (0.016, 0.031, 0.020), skin),
    ]
    scene = trimesh.Scene()
    for part in parts:
        name = part.metadata['name']
        scene.add_geometry(part, node_name=name, geom_name=name)
    scene.metadata.update({
        'title': f'Synthetic stylized {laterality} foot demo',
        'description': 'Illustration for viewer testing only; not a patient scan or clinical model.',
        'laterality': laterality,
        'synthetic': True,
    })
    return scene

models = {}
for side in ('left', 'right'):
    output = OUT_DIR / f'demo-patient-001-{side}-foot.glb'
    output.write_bytes(build_demo_foot(side).export(file_type='glb'))
    models[side] = output
    print(f'{side.title()} foot: {output} ({output.stat().st_size:,} bytes)')

metadata = {
    'patientId': 'demo-patient-001',
    'synthetic': True,
    'records': [
        {'scanId': f'demo-{side}-foot', 'laterality': side,
         'storagePath': f'foot-scans/demo-patient-001/demo-patient-001-{side}-foot.glb',
         'fileFormat': 'model/gltf-binary', 'synthetic': True}
        for side in ('left', 'right')
    ],
}
metadata_path = OUT_DIR / 'demo-patient-001-metadata.json'
metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print('Synthetic metadata:', metadata_path)

## 2. View a generated foot inside this notebook

Drag to rotate and use the mouse wheel or touch gestures to zoom. This interactive preview loads the generated model in the notebook output; it does not send the file to Firebase.

In [ ]:
import base64
from IPython.display import HTML, display

def show_glb_in_notebook(path):
    data = base64.b64encode(Path(path).read_bytes()).decode('ascii')
    data_uri = 'data:model/gltf-binary;base64,' + data
    html = f'''
    <script type="module" src="https://unpkg.com/@google/model-viewer@4.1.0/dist/model-viewer.min.js"></script>
    <div style="font:14px system-ui,sans-serif;color:#172b3a;margin:8px 0">
      <strong>{Path(path).name}</strong> — synthetic demo only
    </div>
    <model-viewer src="{data_uri}" alt="Synthetic foot model for viewer testing"
      camera-controls touch-action="pan-y" shadow-intensity="0.8" exposure="1"
      style="width:100%;height:520px;background:linear-gradient(#fff,#edf4f6);">
    </model-viewer>
    '''
    display(HTML(html))

show_glb_in_notebook(models['right'])  # Change to models['left'] to view the left-foot example.

## 3. Download the synthetic models

Download a generated file to your computer if you want to open it in the Firebase-hosted viewer. The current Firebase viewer selects a file from your device; it does not retrieve this Colab file or save it to Firebase.

In [ ]:
from google.colab import files

# Run one line at a time for the file you want to download.
files.download(str(models['left']))
# files.download(str(models['right']))
# files.download(str(metadata_path))

## 4. Load the Firebase Hosting viewer page in this notebook

This embeds the deployed Hosting page in an iframe. If the page has not been deployed to `coremedi-59bbc` yet, deploy the repository's `public/` site first. In the embedded page, choose a `.glb` file from your computer (download one from the prior section first). Browser security does not let the Firebase page access files that exist only in the Colab runtime.

In [ ]:
from IPython.display import IFrame, display

VIEWER_URL = 'https://coremedi-59bbc.web.app'
display(IFrame(src=VIEWER_URL, width='100%', height=760))

## Notes and limitations
- These are procedurally generated, synthetic illustrations—not 3D scans, accurate anatomy, or clinical measurements.
- The inline preview uses the `model-viewer` library from unpkg and therefore needs internet access.
- This notebook does not upload files to Firebase, connect to Firestore/Storage, identify patients, or calculate clinical risk.
- Follow your organization's privacy, consent, and security rules before working with any real scan. Do not upload PHI to hosted Colab.